In [1]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress
from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")

computed_name = 'ComputedScalarsHelene'
mnt_pth = '/Volumes/FMac/computing/data/antarctica/' #mount path
pth_ismip6hack= mnt_pth + 'ismip6/hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6/hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6/hackathon/ComputedScalarsHelene/'
pth_table = '/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupBMBtodSLR/data_analysis/tables/'

In [2]:
# choose variable,factor and path
var_area = 'iareafl'
var_bmb = 'shelfmelt'

var_dslc = 'dslc_anom'
pth_var_dslc = 'dslc_anom_jb'
factor = -1

fpath = f'{mnt_pth}/ismip6/hackathon/figures/recalc_{var_dslc}_dslcs_dils/'
fpathscat = f'{mnt_pth}/ismip6/hackathon/figures/recalc_{var_dslc}_dslcs_dils_scatter/'
fname = f'cumulative_{var_dslc}_'

In [3]:
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = []
 
metadata_file = 'Metadata.txt'
#df_time = pd.read_csv('tables/time_maximum_BMB.csv')

In [4]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]

In [5]:
models=[]
paths =[]
exps =[]
region_names = ['Amundsen','Ross','FilchnerRonne','Aurora','Wilkes']
dic_sectors ={}
dic_sectors['Amundsen']=np.array([4])
dic_sectors['Ross']=np.array([2,6])
dic_sectors['FilchnerRonne']=np.array([5,11])
dic_sectors['Aurora']=np.array([8])
dic_sectors ['Wilkes']=np.array([7])

renamer = {}
renamer['DOE_MALI_4km']='DOE_MALI_1'
renamer['DOE_MALI_8km_Ant95']='DOE_MALI_2'
renamer['DOE_MALI_8km_AntMean']='DOE_MALI_3'  

for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
            
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
df['AIS']=0
for j in range(1,4):
    df['region_'+str(j)] =0
for key in dic_sectors.keys():
    df[key] = 0
    
df_n = df.copy()
y2sec=-365.25*24*3600
factor = -1

In [6]:
df

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_18,AIS,region_1,region_2,region_3,Amundsen,Ross,FilchnerRonne,Aurora,Wilkes
0,expAE02,DC_ISSM,/Volumes/FMac/computing/data/antarctica/ismip6...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,expAE02,DOE_MALI_4km,/Volumes/FMac/computing/data/antarctica/ismip6...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,expAE02,DOE_MALI_8km_Ant95,/Volumes/FMac/computing/data/antarctica/ismip6...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,expAE02,DOE_MALI_8km_AntMean,/Volumes/FMac/computing/data/antarctica/ismip6...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,expAE02,IGE_ElmerIce,/Volumes/FMac/computing/data/antarctica/ismip6...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,expAE05,VUW_PISM2_s4,/Volumes/FMac/computing/data/antarctica/ismip6...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
184,expAE05,IMAU_UFEMISM1,/Volumes/FMac/computing/data/antarctica/ismip6...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
185,expAE05,IMAU_UFEMISM2,/Volumes/FMac/computing/data/antarctica/ismip6...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
186,expAE05,IMAU_UFEMISM3,/Volumes/FMac/computing/data/antarctica/ismip6...,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
yearmin=0

for i in range (df.shape[0]):
    #load the datasets:
    pth_way = df.iloc[i].Path
    nmname = df.iloc[i].Model
    
    #dslc
    pth_dslc = f'{pth_ismip6hack}/{df.iloc[i].Experiment}/{pth_var_dslc}/computed_{var_dslc}_AIS_{nmname}_{df.iloc[i].Experiment}.nc'
    d3 = xr.open_dataset(pth_dslc,decode_times=False )
    
    #shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/{var_area}/computed_{var_area}_AIS_{nmname}_{df.iloc[i].Experiment}.nc'
    d2 = xr.open_dataset(pth_area,decode_times=False )
    
    #bmb
    if nmname[:4]== 'IMAU':
        pth_bmb = f'{pth_helene}/{df.iloc[i].Experiment}/{var_bmb}/jb_computed_{var_bmb}_AIS_{nmname}_{df.iloc[i].Experiment}.nc'
    else:
        pth_bmb = f'{pth_helene}/{df.iloc[i].Experiment}/{var_bmb}/computed_{var_bmb}_AIS_{nmname}_{df.iloc[i].Experiment}.nc'
    d4 = xr.open_dataset(pth_bmb,decode_times=False )
    #tmin = np.min([d2.time.size,d3.time.size,d4.time.size])
    if yearmin>0:
        tmin = np.max([np.where(d2.time==yearmin)[0][0],0])
        tappend='_tmin' + str(int(d2.time[tmin].values))
    else:
        tmin = 0
        tappend=''
    tmax = np.min([d2.time.size,d3.time.size,d4.time.size])

    ###### start with AIS
    x = np.cumsum(d4[f'{var_bmb}'].values[tmin:tmax].copy()*factor)
    y = -d3[f'{var_dslc}'].values[tmin:tmax].copy()

    if len(x)>1:
        try:
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            df['AIS'].iloc[i]= slope
            df_n['AIS'].iloc[i]= intercept
        except ValueError:
            df['AIS'].iloc[i]= np.nan
            df_n['AIS'].iloc[i]= np.nan
    else:
        df['AIS'].iloc[i]= np.nan
        df_n['AIS'].iloc[i]= np.nan

    
    #all the sectors:
    for j in range(1,19):
        x = np.cumsum(d4[f'{var_bmb}_sector_'+str(j)].values[tmin:tmax].copy()*factor)
        y = -d3[f'{var_dslc}_sector_'+str(j)].values[tmin:tmax].copy()
        
        mnan1 = np.isnan(y)
        mnan2 = np.isnan(x)
        mnan = mnan1 | mnan2
    
        if len(x[~mnan])>1:
            try:
                slope, intercept, r_value, p_value, std_err = linregress(x[~mnan], y[~mnan])
                df['sector_'+str(j)].iloc[i] = slope
                df_n['sector_'+str(j)].iloc[i] = intercept
            except ValueError:
                df['sector_'+str(j)].iloc[i] = np.nan
                df_n['sector_'+str(j)].iloc[i] = np.nan
        else:
            df['sector_'+str(j)].iloc[i] = np.nan
            df_n['sector_'+str(j)].iloc[i] = np.nan
            
    #all the regions
    for j in range(1,4):
        x = np.cumsum(d4[f'{var_bmb}_region_'+str(j)].values[tmin:tmax].copy()*factor)
        y = -d3[f'{var_dslc}_region_'+str(j)].values[tmin:tmax].copy()
        
        mnan1 = np.isnan(y)
        mnan2 = np.isnan(x)
        mnan = mnan1 | mnan2

        if len(x[~mnan])>1:
            try:
                slope, intercept, r_value, p_value, std_err = linregress(x[~mnan], y[~mnan])
                df['region_'+str(j)].iloc[i] = slope
                df_n['region_'+str(j)].iloc[i] = intercept
            except ValueError:
                df['region_'+str(j)].iloc[i] = np.nan
                df_n['region_'+str(j)].iloc[i] = np.nan
        else:
            df['region_'+str(j)].iloc[i] = np.nan
            df_n['region_'+str(j)].iloc[i] = np.nan
            
    #specific region,combination of sectors: Amundsen, Ross, FRIS
    for key in dic_sectors.keys():       
        # create ys areas etc
        the_xs = np.zeros((len(dic_sectors[key]),tmax-tmin))
        the_ys = np.zeros((len(dic_sectors[key]),tmax-tmin))
        for k,j in enumerate(dic_sectors[key]):
            the_ys[k,:]= -d3[f'{var_dslc}_sector_'+str(j)].values[tmin:tmax].copy()
            the_xs[k,:]= d4[f'{var_bmb}_sector_'+str(j)].values[tmin:tmax].copy()*factor
            #the_ys[k,:] =d['shelfmelt_sector_'+str(j)].values[tmin:tmax].copy()* factor
            #the_areas[k,:]=d2['iareafl_sector_'+str(j)].values[tmin:tmax].copy()
            #the_tfs[k,:]=d3['thermalforcing_sector_'+str(j)].values[tmin:tmax].copy()*the_areas[k,:].copy()
        y = np.sum(the_ys, axis=0)
        x = np.cumsum(np.sum(the_xs, axis = 0)).squeeze() #gives average thermal foricng line
        mnan1 = np.isnan(y)
        mnan2 = np.isnan(x)
        mnan = mnan1 | mnan2

        if len(x[~mnan])>1:
            try:
                print(f'some nans!{i} at {df.iloc[i].Model}')
                slope, intercept, r_value, p_value, std_err = linregress(x[~mnan], y[~mnan])
                df[key].iloc[i] = slope
                df_n[key].iloc[i] = intercept
            except ValueError:
                df[key].iloc[i] = np.nan
                df_n[key].iloc[i] = np.nan
        else:
            print('pnli nans',df.iloc[i].Model)
            df[key].iloc[i] = np.nan
            df_n[key].iloc[i] =np.nan

df.to_csv(f'tables/linear_dslc_sensitivity_from_cumBMB2{tappend}.csv', index=False)
df_n.to_csv(f'tables/linear_dslc_sensitivity_from_cumBMB_intercept2{tappend}.csv', index=False)

/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_12639/3990679852.py:38: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['AIS'].iloc[i]= slope
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_12639/3990679852.py:39: Futu

some nans!0 at DC_ISSM
some nans!0 at DC_ISSM
some nans!0 at DC_ISSM
some nans!0 at DC_ISSM
some nans!0 at DC_ISSM
some nans!1 at DOE_MALI_4km
some nans!1 at DOE_MALI_4km
some nans!1 at DOE_MALI_4km
some nans!1 at DOE_MALI_4km
some nans!1 at DOE_MALI_4km
some nans!2 at DOE_MALI_8km_Ant95
some nans!2 at DOE_MALI_8km_Ant95
some nans!2 at DOE_MALI_8km_Ant95
some nans!2 at DOE_MALI_8km_Ant95
some nans!2 at DOE_MALI_8km_Ant95
some nans!3 at DOE_MALI_8km_AntMean
some nans!3 at DOE_MALI_8km_AntMean
some nans!3 at DOE_MALI_8km_AntMean
some nans!3 at DOE_MALI_8km_AntMean
some nans!3 at DOE_MALI_8km_AntMean
some nans!4 at IGE_ElmerIce
some nans!4 at IGE_ElmerIce
some nans!4 at IGE_ElmerIce
some nans!4 at IGE_ElmerIce
some nans!4 at IGE_ElmerIce
some nans!5 at ILTS_SICOPOLIS
some nans!5 at ILTS_SICOPOLIS
some nans!5 at ILTS_SICOPOLIS
some nans!5 at ILTS_SICOPOLIS
some nans!5 at ILTS_SICOPOLIS
some nans!6 at IMAU_UFEMISM1
some nans!6 at IMAU_UFEMISM1
some nans!6 at IMAU_UFEMISM1
some nans!6 at IMA

In [13]:
df

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_18,AIS,region_1,region_2,region_3,Amundsen,Ross,FilchnerRonne,Aurora,Wilkes
0,expAE02,DC_ISSM,/Volumes/FMac/computing/data/antarctica/ismip6...,6.800337e-07,0.000003,0.000004,0.000005,0.000006,3.613986e-07,0.000006,...,2.523831e-06,0.000004,0.000005,0.000004,-0.000002,0.000005,0.000003,0.000007,0.000005,0.000006
1,expAE02,DOE_MALI_4km,/Volumes/FMac/computing/data/antarctica/ismip6...,2.303872e-05,0.000008,0.000014,0.000020,0.000010,8.492191e-07,0.000022,...,2.308598e-05,0.000013,0.000013,0.000012,0.000015,0.000020,0.000006,0.000011,0.000014,0.000022
2,expAE02,DOE_MALI_8km_Ant95,/Volumes/FMac/computing/data/antarctica/ismip6...,1.885100e-05,0.000015,0.000014,0.000013,0.000014,7.584846e-07,0.000026,...,1.890107e-05,0.000015,0.000014,0.000015,0.000013,0.000013,0.000013,0.000014,0.000018,0.000026
3,expAE02,DOE_MALI_8km_AntMean,/Volumes/FMac/computing/data/antarctica/ismip6...,1.674963e-05,0.000012,0.000014,0.000015,0.000012,6.646679e-07,0.000027,...,1.803675e-05,0.000014,0.000014,0.000013,0.000012,0.000015,0.000010,0.000013,0.000017,0.000027
4,expAE02,IGE_ElmerIce,/Volumes/FMac/computing/data/antarctica/ismip6...,-8.845767e-07,0.000002,0.000004,0.000005,0.000004,4.397595e-07,0.000012,...,5.308800e-07,0.000003,0.000003,0.000004,-0.000007,0.000005,0.000001,0.000004,0.000004,0.000012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,expAE05,VUW_PISM2_s4,/Volumes/FMac/computing/data/antarctica/ismip6...,6.174647e-06,0.000022,0.000013,0.000024,0.000017,1.487945e-05,0.000027,...,2.381517e-05,0.000022,0.000021,0.000025,0.000012,0.000024,0.000022,0.000019,0.000020,0.000027
184,expAE05,IMAU_UFEMISM1,/Volumes/FMac/computing/data/antarctica/ismip6...,NaN,0.000118,NaN,NaN,0.000016,NaN,NaN,...,-1.106204e-04,0.000026,0.000041,0.000008,-0.000227,NaN,0.000114,0.000017,0.001019,NaN
185,expAE05,IMAU_UFEMISM2,/Volumes/FMac/computing/data/antarctica/ismip6...,NaN,0.000135,NaN,NaN,0.000023,NaN,NaN,...,NaN,0.000035,0.000054,0.000005,-0.000323,NaN,0.000133,0.000023,-0.000781,NaN
186,expAE05,IMAU_UFEMISM3,/Volumes/FMac/computing/data/antarctica/ismip6...,-3.582389e-05,0.000097,NaN,0.000230,0.000024,3.612398e-04,-0.000141,...,NaN,0.000041,0.000044,0.000038,-0.000104,0.000230,0.000097,0.000025,0.000081,-0.000141


In [27]:
str(int(d2.time[tmin].values))

'2200'